# Day three — Species distribution modeling

---

-> summary: we have records of where species have been observed, but those points are not yet an explanation. Where is the species found? Where is it absent, or simply not recorded? What environmental conditions might explain the pattern? Once we have an idea of what determines the presence, absence, or even abundance of a species, we can go beyond observation, and model the species’ distribution. With such models, we can predict where we have no data, identify key regions for conservation, and even model what could happen if the environment were to change.

- PART I: From observations to ecological questions.
- PART II: Identifying environmental predictors: climate, land use, and habitat suitability.
- PART III: Species distribution modeling.
- PART IV: How to be a good scientist: tips, tricks, and pitfalls.


## *PART I: From observations to ecological questions*

## 📋 Planning note (instructor-facing — delete before making the student `day3.ipynb`)

**Target species: Philippine tarsier (*Carlito syrichta*).** Endemic, IUCN Near Threatened, and — unlike
the earlier flying-fox / eagle candidates — genuinely climate/habitat-restricted rather than either
ubiquitous (*Macaca fascicularis*) or archipelago-wide (*Pteropus vampyrus*), which should give the
model a real signal to find. Lives inside the broader `gbif_mammals_philippines.csv` compilation
(no dedicated single-species pull like Day 2's flying fox/eagle): 408 records within the PH bbox, 312
unique ~100 m locations, spanning 6 islands / 27 provinces. Two data-quality findings from checking this
before committing (see below — both become teaching material, not just cleanup):
- A dense cluster on Bohol (~124°E, 9.8°N) coincides with the Tarsier Sanctuary / tourism sites — likely
  captive or semi-captive individuals, not independent wild observations.
- **6 records fall well outside the species' known native range** (the Mindanao faunal region: Mindanao,
  Bohol, Leyte, Samar, and nearby islands) — one near 15°N in central Luzon, three around
  121.7–122°E/12.7–12.9°N (Mindoro/Romblon). Almost certainly zoo/captive records mislabeled as wild;
  these get filtered out in Part I rather than fed to the model.

**Data inventory & feasibility (updated):**

| Data | Resolution | Extent | Status |
|---|---|---|---|
| Tarsier occurrences | point | Philippines-wide | ✅ have — filter `gbif_mammals_philippines.csv` to `species == "Carlito syrichta"` |
| Köppen-Geiger classification | 1 km (~0.0083°) | Philippines-wide | ✅ have (`kg_philippines_present.nc` + `..._future_ssp460.nc`) |
| WorldClim BIO5 (max temp of warmest month) & BIO14 (precip of driest month) | 30 arc-sec (~1 km) | Philippines-wide | ✅ have (`worldclim_bio_philippines_1970_2000.nc`) — 1970–2000 baseline, the only period WorldClim offers at this resolution |
| Elevation (Copernicus DEM GLO-30, aggregated) | 1 km | Philippines-wide | ✅ have (`dem_philippines_1km.nc`) — native 30 m block-averaged onto the same grid, no raw 30 m file kept |
| Land-sea mask | 0.25° / 0.1° / 1 km | Philippines-wide | ✅ have (3 resolutions) |
| Future climate (ERA5-derived, SSP4-6.0) | 0.1° (~11 km) | Philippines-wide | ✅ have (`climate_philippines_2071_2099_ssp460_mean.nc`) — **different lineage and coarser than the 1 km present-day layers above; see capstone note in Part III** |
| Forest cover (Hansen) / satellite imagery | 30 m / ~100–200 m | **Negros only** | ⚠️ partial — unchanged from Day 2, still an optional regional stretch only |

**Resolution strategy (upgraded from the original plan):** KG, WorldClim BIO5/BIO14, the land-sea mask,
and the DEM are all now aligned to the *same* ~1 km national grid (2040 × 1620 cells,
lon 113.5–127.0°, lat 4.5–21.5°) — see `behind-the-scenes/day3_bts.ipynb` §1–2. That's a real upgrade
over the original plan (which would have capped the national model at the 0.1° ERA5 climatology): the
core model now runs at ~1 km nationally without needing the Negros-only Hansen/Sentinel layers at all.

**Watch out for: the bbox is a rectangle, not the Philippines.** Because it's a plain lon/lat box, the
grid's southwest corner (roughly lon < 119°, lat < 7.5°) includes a real slice of northern Borneo
(Sabah) — confirmed by the DEM's max elevation (~3900 m) matching Mt. Kinabalu, not any Philippine peak.
Presence points are unaffected (GBIF's own `country=PH` filter already restricts those), but **background
/ pseudo-absence sampling must exclude that corner explicitly**, or the model quietly learns from Bornean
climate. No new dependency needed (no shapefile/geopandas) — a hand-specified sub-box exclusion is
sufficient and stays in the workshop's "homemade" style. This doubles as a concrete Part IV pitfall.

**Scope note:** Day 3 should end up *shorter* than Day 2, so Parts II–III stay lean. Loading three
pre-aligned, ready-to-use national layers (instead of deriving bioclim-style summaries from monthly
ERA5 grids by hand) actually helps here — that derivation exercise is dropped from Day 3 since it's no
longer how the core predictors are produced (it's already a Day 1 skill; no need to repeat it). This is
also the first time students see a real fitted statistical model rather than a hand-coded rule tree
(Day 1/2's "homemade classifiers") — a deliberate step up, not a break in style. Part IV should stay short
discussion, not new heavy code.

In [ ]:
# 1
import os
if not os.path.exists("/content/USLS-Workshop"):
    !git clone "https://github.com/Cas-Dec/USLS-Workshop.git" /content/USLS-Workshop
%cd /content/USLS-Workshop
!pip install -q .

#from google.colab import drive
#drive.mount('/content/drive')

**Plan.** Goal: get students to look critically at presence-only occurrence data *before* jumping to
modeling — and this time the cleaning isn't hypothetical, it's two real findings worth walking through.

1. Filter `gbif_mammals_philippines.csv` (Day 2's broader mammal compilation) down to
   `species == "Carlito syrichta"` and the PH bbox — unlike the flying fox/eagle, the tarsier doesn't
   have its own dedicated pre-cleaned file, so this filtering step is new.
2. Plot presence points over a Philippines outline (reuse the land-sea-mask contour pattern from Day 1/2
   maps). Prompt: does the species look randomly scattered, or clustered? Two things should jump out:
   - A dense cluster on Bohol — home to the Tarsier Sanctuary and other tourist-facing sites. Discuss:
     is this real ecological signal, or repeated observation of a small number of captive/habituated
     individuals?
   - A handful of points far outside the species' known native range (the Mindanao faunal region —
     Mindanao, Bohol, Leyte, Samar, and nearby islands): one near central Luzon, a few around
     Mindoro/Romblon. Almost certainly zoo records mislabeled as wild.
3. Have students filter out the out-of-range points (a simple lat/lon rule is enough — no need for
   anything fancier) before moving on. This is the "clean, model-ready" presence set. The Bohol cluster
   is *not* removed (it's still real habitat), but flag it as a caveat to revisit in Part IV.
4. Discussion, pushing Day 2's sampling-bias theme specifically toward what it means *for modeling*:
   - GBIF gives us presences, not absences — an unrecorded area might be truly unsuitable, or just
     unsurveyed (near no roads/observers). We can't tell the difference from the points alone.
   - Introduce **background points / pseudo-absences** conceptually here (needed before Part III can fit
     anything): without real absences, we approximate them by sampling random points across the study
     area.
5. Land on the specific question Parts II-III will answer: *"Which combination of temperature,
   precipitation, and elevation is associated with where Philippine tarsiers have been recorded, and can
   that predict suitability elsewhere in the country?"*

### Load and filter occurrence records

Unlike the flying fox/eagle in Day 2, the tarsier doesn't have its own dedicated pre-cleaned file —
pull it out of the broader `gbif_mammals_philippines.csv` compilation from Day 2 Part II (all
present-day terrestrial mammal records for the Philippines).

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from workshop_utils import PROCESSED_DIR

mammals = pd.read_csv("../data/processed/day2/gbif_mammals_philippines.csv", low_memory=False)

tarsier_raw = mammals[mammals['species'] == 'Carlito syrichta'].dropna(
    subset=['decimalLongitude', 'decimalLatitude'])
tarsier_raw = tarsier_raw[
    tarsier_raw['decimalLongitude'].between(113.5, 127.0) &
    tarsier_raw['decimalLatitude'].between(4.5, 21.5)
]
print(f'Raw records: {len(tarsier_raw)}')
tarsier_raw.head()

### Look at the raw points before doing anything else

Plot them over a Philippines outline (same land-sea-mask contour trick from Day 1/2). Does the
species look randomly scattered, or clustered?

In [ ]:
lsm = xr.open_dataset(PROCESSED_DIR / 'land-sea-mask_0p00833333.nc')
land = lsm['land_sea_mask'].values > 0.5
LSM_LAT, LSM_LON = lsm['lat'].values, lsm['lon'].values

fig, ax = plt.subplots(figsize=(7, 9))
ax.contourf(LSM_LON, LSM_LAT, land, levels=[0.5, 1], colors=['#d9d9d9'])
ax.scatter(tarsier_raw['decimalLongitude'], tarsier_raw['decimalLatitude'], s=10, color='crimson')

ax.set_xlim(LSM_LON.min(), LSM_LON.max())
ax.set_ylim(LSM_LAT.min(), LSM_LAT.max())
ax.set_aspect('equal')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Philippine tarsier — {len(tarsier_raw)} raw GBIF records')

Two things should jump out:
- A dense cluster on Bohol (~124°E, 9.8°N) — home to the Tarsier Sanctuary and other tourist-facing
  sites. Is this real ecological signal, or repeated observation of a small number of
  captive/habituated individuals?
- A handful of points far outside the species' known native range (the Mindanao faunal region —
  Mindanao, Bohol, Leyte, Samar, and nearby islands): one near central Luzon, a few around
  Mindoro/Romblon. Almost certainly zoo records mislabeled as wild.

In [ ]:
# Native range = the Mindanao faunal region (Mindanao, Bohol, Leyte, Samar, nearby islands). The
# handful of points well outside it (central Luzon; Mindoro/Romblon) are almost certainly captive/zoo
# records mislabeled as wild -- drop them. The Bohol cluster is NOT removed here (it's still real
# habitat) -- flagged as a sampling-bias caveat to revisit in Part IV.
out_of_range = (
    (tarsier_raw['decimalLatitude'] > 13.5) |
    ((tarsier_raw['decimalLongitude'] < 122.5) & (tarsier_raw['decimalLatitude'] > 11.5))
)
print(f'Dropping {int(out_of_range.sum())} likely captive/out-of-range records')
tarsier = tarsier_raw[~out_of_range].copy()

# Remove spatial duplicates rounded to 0.01° (~1 km) -- the model will run on a 1 km grid anyway, so
# records that would land on the same grid cell carry no extra information for it
tarsier['lon_r'] = tarsier['decimalLongitude'].round(2)
tarsier['lat_r'] = tarsier['decimalLatitude'].round(2)
tarsier = tarsier.drop_duplicates(subset=['lon_r', 'lat_r'])
tarsier = tarsier.rename(columns={'decimalLongitude': 'lon', 'decimalLatitude': 'lat'})[['lon', 'lat']]

print(f'Clean presence set: {len(tarsier)} points')
tarsier.head()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
ax.contourf(LSM_LON, LSM_LAT, land, levels=[0.5, 1], colors=['#d9d9d9'])

ax.scatter(tarsier['lon'], tarsier['lat'], s=10, color='crimson', label=f'Kept (n={len(tarsier)})')
dropped = tarsier_raw[out_of_range]
ax.scatter(dropped['decimalLongitude'], dropped['decimalLatitude'], s=50, facecolor='none',
           edgecolor='orange', linewidths=1.5, label=f'Dropped, out of range (n={len(dropped)})')

ax.set_xlim(LSM_LON.min(), LSM_LON.max())
ax.set_ylim(LSM_LAT.min(), LSM_LAT.max())
ax.set_aspect('equal')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Philippine tarsier — cleaned presence set')
ax.legend(loc='lower left', frameon=False)

### Presence-only data isn't the whole picture

GBIF gives us presences, not absences — an unrecorded area might be truly unsuitable, or just
unsurveyed (no roads, no observers nearby). We can't tell the difference from the points alone. To fit
a model in Part III we need something to contrast presences against: **background points**
(pseudo-absences). Without real absence surveys, we approximate them by sampling random points across
the study area — Part II builds these.

**The question Parts II–III will answer:** which combination of temperature, precipitation, and
elevation is associated with where Philippine tarsiers have been recorded, and can that predict
suitability elsewhere in the country?

## *PART II: Identifying environmental predictors: climate, land use, and habitat suitability*

**Plan.** Goal: end up with one clean table — one row per point (presence + background), one column per
environmental variable. Simpler than originally planned: the predictor layers are now already
pre-aligned to one shared 1 km grid (`behind-the-scenes/day3_bts.ipynb` §1–3), so there's no need to
derive bioclim-style summaries from monthly climate grids by hand (that's a Day 1 skill already covered;
no need to repeat it) — this is one of the things that keeps Day 3 shorter than Day 2.

1. Load the four aligned national layers directly: `kg_philippines_present.nc` (categorical climate
   zone), `worldclim_bio_philippines_1970_2000.nc` (`bio5`, `bio14`), `dem_philippines_1km.nc`
   (`elevation`), and `forest_cover_philippines_1km.nc` (`treecover2000`). All four share the exact same
   `lat`/`lon` grid.
2. Generate background/pseudo-absence points: random lon/lat draws over land only (reject using the 1 km
   land-sea mask), roughly matching the cleaned presence count. **Important:** also reject any draw
   inside the bbox's Borneo corner (roughly lon < 119°, lat < 7.5°) — the rectangular study area spills
   into Sabah, Malaysia, and the land-sea mask alone won't catch that (it only knows land vs. ocean, not
   country). A hand-specified sub-box exclusion is enough; no shapefile/geopandas needed.
3. Extract each predictor's value at every presence + background point. Because all four rasters now
   share one grid, this only needs *one* nearest-grid-cell index lookup per point (the
   `np.argmin(np.abs(array - value))` trick from Day 2's KG work), reused across `kg_class`, `bio5`,
   `bio14`, `elevation`, and `treecover2000` — a nice payoff from aligning the data up front instead of
   per-layer.
4. Assemble into one `pandas.DataFrame`: `lon, lat, presence (0/1), kg_class, bio5, bio14, elevation,
   treecover2000`.

**Update, superseding the "optional stretch" below:** the climate-only model (Part III) couldn't recover
the tarsier's known range, so national forest cover got promoted from a Negros-only stretch goal to a
core predictor — see `behind-the-scenes/day3_bts.ipynb` §3. `download_hansen_tiles()`'s decimated-read
sibling `build_hansen_mosaic()` extends the same Day 2 data source to the full country. The *original*
optional stretch — an even finer-resolution Negros-only zoom using Day 2's Hansen/land-cover classifier
directly (not aggregated to 1 km) — is still on the table as a genuinely separate, still-cut-for-time
idea, since Negros is at the edge of, not central to, the tarsier's native range.

### Load the four aligned predictor layers

`kg_philippines_present.nc` (categorical climate zone), `worldclim_bio_philippines_1970_2000.nc`
(`bio5`, `bio14`), `dem_philippines_1km.nc` (`elevation`), and `forest_cover_philippines_1km.nc`
(`treecover2000`, national Hansen canopy cover — extended from Day 2's Negros-only pull) are all
pre-aligned to one shared 1 km grid (see `behind-the-scenes/day3_bts.ipynb`) — no derivation needed,
just load and go.

In [ ]:
kg_grid   = xr.open_dataset('../data/processed/kg_philippines_present.nc')
bio_ds    = xr.open_dataset('../data/processed/worldclim_bio_philippines_1970_2000.nc')
dem_ds    = xr.open_dataset('../data/processed/dem_philippines_1km.nc')
forest_ds = xr.open_dataset('../data/processed/forest_cover_philippines_1km.nc')

PH_LAT, PH_LON = kg_grid['lat'].values, kg_grid['lon'].values
print(f'Shared grid: {len(PH_LAT)} x {len(PH_LON)} @ ~1 km -- same grid as the land-sea mask above')
assert (PH_LAT == LSM_LAT).all() and (PH_LON == LSM_LON).all()

### Generate background (pseudo-absence) points

Random points from Philippine land area, using the same 1 km land-sea mask from Part I. **Watch out:**
the rectangular PH bbox spills into northern Borneo (Sabah) — confirmed later by the DEM's ~3900 m max
elevation matching Mt. Kinabalu, not any Philippine peak — so that corner needs excluding explicitly,
or the model would quietly learn from a foreign climate.

In [ ]:
# Exclude the bbox's Borneo corner (roughly lon < 119, lat < 7.5) from the background pool -- the
# rectangular study area spills into Sabah, Malaysia, and the land-sea mask alone won't catch that
# (it only knows land vs. ocean, not country). See the planning note above.
lon_grid, lat_grid = np.meshgrid(PH_LON, PH_LAT)
borneo_corner = (lon_grid < 119.0) & (lat_grid < 7.5)
land_mask = land & ~borneo_corner

land_rows, land_cols = np.where(land_mask)
print(f'Land pixels available (Philippines only, Borneo corner excluded): {len(land_rows):,}')

n_pseudo = min(len(tarsier) * 10, 2000)   # up to 10x presence count, capped at 2000
rng = np.random.default_rng(42)
idx = rng.choice(len(land_rows), size=n_pseudo, replace=False)
pseudo_df = pd.DataFrame({
    'lon': PH_LON[land_cols[idx]],
    'lat': PH_LAT[land_rows[idx]],
})

print(f'Generated {len(pseudo_df)} pseudo-absence points')

### Extract predictor values

`kg_philippines_present.nc`, `worldclim_bio_philippines_1970_2000.nc`, and `dem_philippines_1km.nc`
all share the exact same 1 km grid as the land-sea mask above — so the nearest-grid-cell lookup (the
`np.argmin(np.abs(array - value))` trick from Day 2's KG work) only needs to happen **once** per
point, then gets reused across all four layers.

In [ ]:
def nearest_rowcol(lons, lats, grid_lon=PH_LON, grid_lat=PH_LAT):
    """Nearest-grid-cell index for every (lon, lat) point, vectorised over the whole grid at once."""
    col_idx = np.abs(grid_lon[None, :] - np.asarray(lons)[:, None]).argmin(axis=1)
    row_idx = np.abs(grid_lat[None, :] - np.asarray(lats)[:, None]).argmin(axis=1)
    return row_idx, col_idx


def extract_predictors(df):
    row_idx, col_idx = nearest_rowcol(df['lon'].values, df['lat'].values)
    out = df.copy()
    out['kg_class']      = kg_grid['kg_class'].values[row_idx, col_idx]
    out['bio5']          = bio_ds['bio5'].values[row_idx, col_idx]
    out['bio14']         = bio_ds['bio14'].values[row_idx, col_idx]
    out['elevation']     = dem_ds['elevation'].values[row_idx, col_idx]
    out['treecover2000'] = forest_ds['treecover2000'].values[row_idx, col_idx]
    return out


print('Extracting values at presence points ...')
tarsier_out = extract_predictors(tarsier)
tarsier_out['presence'] = 1

print('Extracting values at pseudo-absence points ...')
pseudo_out = extract_predictors(pseudo_df)
pseudo_out['presence'] = 0

sdm_df = pd.concat([tarsier_out, pseudo_out], ignore_index=True).dropna()
n_pres = int(sdm_df['presence'].sum())
n_abs  = int((sdm_df['presence'] == 0).sum())
print(f'\nSDM dataset: {n_pres} presences + {n_abs} background points')
sdm_df.describe()

### Save the SDM-ready table

One row per point (presence + background), one column per environmental variable — ready for Part III.

In [ ]:
day3_dir = Path('../data/processed/day3')
day3_dir.mkdir(parents=True, exist_ok=True)
out_path = day3_dir / 'sdm_tarsier.csv'

if not os.path.exists(out_path):
    sdm_df.to_csv(out_path, index=False)
    print(f'Saved: {out_path}')

## *PART III: Species distribution modeling*

**Plan.** Goal: fit and visualize a simple, homemade presence/background model — the same "rule → function
→ grid → map" pattern from Day 1 (climate classifier) and Day 2 (land-cover classifier), now applied to a
genuinely *predictive* (not just descriptive) use case, and now genuinely national at ~1 km rather than
0.1°.

1. Split Part II's table into predictors (`X`: `bio5`, `bio14`, `elevation`, optionally `kg_class`) and
   presence label (`y`).
2. Fit `sklearn.linear_model.LogisticRegression` (already in `environment.yml`, unused so far) on `X`/`y` —
   the simplest workable presence/background SDM. Deliberately not a dedicated SDM package, consistent with
   the workshop's homemade-first philosophy; this is the first time students fit a real statistical model
   rather than hand-coding a decision rule.
3. Predict suitability (predicted probability) across every land pixel of the shared 1 km grid — excluding
   the Borneo corner, same as the background sampling in Part II — to get a Philippines-wide habitat
   suitability map at real 1 km resolution.
4. Plot it with the real (cleaned) presence points overlaid — sanity-check by eye: does the high-suitability
   zone concentrate around the Mindanao faunal region (Mindanao, Bohol, Leyte, Samar), matching the species'
   documented range? This replaces the congener-species validation trick from `example_study.pdf` (the
   tarsier has no close congener in the Philippines to compare against) with an equally simple check against
   the species' independently-known range.
5. **Capstone, using data we already have — with an honest caveat:** re-predict the same fitted model onto
   `climate_philippines_2071_2099_ssp460_mean.nc` for a future suitability map. Flag explicitly before doing
   this: that future layer is ERA5-derived at 0.1° (~11 km), a **different product and resolution** from the
   1 km WorldClim 1970–2000 baseline used to train the model — so BIO5/BIO14-equivalent values need to be
   derived from it and then either resampled onto the 1 km grid or accepted at their native coarser
   resolution for this one map. Don't hide this mismatch from students — it's the cleanest real example of
   the "different data lineage" extrapolation pitfall for Part IV.
6. **Optional stretch:** zoom the same suitability map into the Negros bbox and overlay Day 2's land-cover
   classifier / Hansen forest-cover output — does finer-resolution data change the local picture? Note
   Negros is at the edge of, not central to, the tarsier's native range, so this is more a resolution-contrast
   demo than a core validation. Time-permitting only.

**Update after actually running this:** step 4's sanity check failed — climate + elevation alone (AUC
~0.58) put the brightest suitability in Palawan/the Luzon highlands, not the tarsier's real range. Rather
than accept that quietly, we went back to Part II and added national forest cover as a fourth predictor
(see the update note there). It nudged the AUC to ~0.61 and added real within-region texture, but did
**not** fix the national mismatch — Palawan still out-scores Mindanao/Bohol/Leyte/Samar. That negative
result is now the point: it's a clean, concrete demonstration that unmodeled biogeography/dispersal
history, not missing environmental variables, is the actual limit here. See the markdown right after the
present-day map for the full discussion.

### Fit the model

A simple presence/background logistic regression on the four continuous predictors from Part II —
deliberately not a dedicated SDM package, consistent with the workshop's homemade-first philosophy.
This is the first time in the workshop you fit a real statistical model rather than hand-coding a
decision rule.

In [ ]:
FEATURES = ['bio5', 'bio14', 'elevation', 'treecover2000']
X = sdm_df[FEATURES].values
y = sdm_df['presence'].values

model = LogisticRegression(max_iter=1000)
model.fit(X, y)

probs = model.predict_proba(X)[:, 1]
print('Coefficients:')
for name, coef in zip(FEATURES, model.coef_[0]):
    print(f'  {name:>13s}: {coef:+.4f}')
print(f'  {"intercept":>13s}: {model.intercept_[0]:+.4f}')
print(f'\nAUC: {roc_auc_score(y, probs):.3f}  (0.5 = no better than random, 1.0 = perfect separation)')

Adding forest cover nudges the AUC from ~0.58 (climate + elevation alone) to ~0.61, and the
coefficient on `treecover2000` comes out positive — ecologically sensible (more canopy, more
suitable) rather than an artifact. Still only modestly better than random, though. Keep that in mind
when looking at the map below.

### Predict suitability across the Philippines

Apply the fitted model to every land pixel of the shared 1 km grid — excluding the bbox's Borneo
corner, same as the background sampling in Part II — to get a Philippines-wide habitat suitability
map at real 1 km resolution.

In [ ]:
rows, cols = np.where(land_mask)
X_grid = np.column_stack([
    bio_ds['bio5'].values[rows, cols],
    bio_ds['bio14'].values[rows, cols],
    dem_ds['elevation'].values[rows, cols],
    forest_ds['treecover2000'].values[rows, cols],
])
valid = ~np.isnan(X_grid).any(axis=1)   # a few coastal cells fall on WorldClim/KG nodata -- see Part IV

suitability = np.full(land_mask.shape, np.nan, dtype='float32')
suitability[rows[valid], cols[valid]] = model.predict_proba(X_grid[valid])[:, 1]

print(f'Predicted {valid.sum():,} land cells ({(~valid).sum()} skipped, no predictor data)')
print(f'Suitability range: {np.nanmin(suitability):.3f} to {np.nanmax(suitability):.3f}')

### Plot the suitability map

Overlay the real (cleaned) presence points — does the high-suitability zone concentrate around the
tarsier's documented native range (the Mindanao faunal region: Mindanao, Bohol, Leyte, Samar)? That's
the sanity check `example_study.pdf`'s congener trick would normally provide, adapted here since the
tarsier has no close congener in the Philippines to compare against.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
im = ax.pcolormesh(PH_LON, PH_LAT, suitability, cmap='viridis', shading='auto')
plt.colorbar(im, ax=ax, label='Predicted suitability', shrink=0.7)

presence_pts = sdm_df[sdm_df['presence'] == 1]
ax.scatter(presence_pts['lon'], presence_pts['lat'], s=8, color='red', label='Tarsier presence')

ax.set_xlim(PH_LON.min(), PH_LON.max())
ax.set_ylim(PH_LAT.min(), PH_LAT.max())
ax.set_aspect('equal')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Predicted habitat suitability — Philippine tarsier (present-day)')
ax.legend(loc='lower left', frameon=False)

### Is our model really that bad?

Before writing this off as a failed model — is a modest AUC the same thing as a *wrong* model? The
map says Palawan's climate and forest cover look at least as tarsier-friendly as Mindanao's. If GBIF
told us tomorrow that tarsiers actually do live in Palawan, this same model would suddenly look
pretty good. It doesn't say that — so what does the low AUC actually tell us: that the model got the
environmental relationship wrong, or that environmental suitability was never the whole story for
where this species lives?

**It's the second one — and there's a well-documented reason.** Palawan has forest and a climate not
unlike parts of Mindanao, but no tarsiers, because of geological history rather than present-day
environment.

During Pleistocene glacial periods, global sea level dropped by roughly 120 m at times, exposing land
that's underwater today and fusing many current-day islands into a few much larger ones —
**Pleistocene Aggregate Island Complexes (PAICs)**, a framework established by Heaney (1985, 1986)
that still organizes how Philippine biogeography is understood. Four PAICs matter here: Greater
Luzon, Greater Mindanao (Mindanao + Bohol + Leyte + Samar + Basilan), Greater Negros–Panay, and
Greater Palawan. Crucially, **Palawan is the exception**: at its lowest sea-level stand it connected
to Borneo and mainland Sundaland, not to the rest of the Philippines — its fauna is more Bornean than
Philippine. The other three PAICs were never connected to each other, or to Palawan, even at the
lowest sea levels.

The Philippine tarsier's entire native range is the Greater Mindanao PAIC. It was never connected to
Palawan (Sundaic) or to Luzon (its own separate PAIC) by dry land at any point — so it never had the
opportunity to disperse there, regardless of whether the climate or forest there would have suited
it. That's a **water barrier**, not a climate one, and no combination of `bio5` / `bio14` /
`elevation` / `treecover2000` can encode it, because none of those variables were ever about
geographic connectivity in the first place.

**The general lesson:** a species distribution model estimates environmental *suitability*, not
*realized distribution*. The two agree wherever dispersal and history haven't gotten in the way, and
diverge wherever they have. A low AUC here isn't (only) a modeling failure — it's a rough measurement
of how much of this species' range is explained by environment alone, and the honest answer is "not
that much, once geography has already done most of the work."

*Sources: Heaney, L. R. (1985). Zoogeographic evidence for middle and late Pleistocene land bridges to
the Philippine Islands. Modern Quaternary Research in Southeast Asia. Heaney, L. R. (1986).
Biogeography of mammals in SE Asia: estimates of rates of colonization, extinction, and speciation.
Biological Journal of the Linnean Society.*

### Capstone — projected suitability, 2071–2099 (SSP4-6.0)

Re-predict the fitted model onto Day 1's future climate file. **An honest caveat first:**
`climate_philippines_2071_2099_ssp460_mean.nc` comes from the Beck et al. ensemble at 0.1° (~11 km) —
a **different product, and a coarser resolution**, than the 1 km WorldClim 1970–2000 baseline the
model was actually trained on. We derive BIO5/BIO14-equivalents from it directly (max/min across its
12 monthly values) and block-average the 1 km DEM down to match its grid. The future map will
visibly look blockier than the present one purely because of this — not because elevation itself got
coarser.

**A second, stronger caveat for forest cover specifically:** there is no "2071–2099 forest cover"
layer, so the capstone reuses today's `treecover2000` unchanged — implicitly assuming forest cover
stays exactly as it is now for the next ~50-75 years. Unlike elevation, that's a real, checkable
assumption this dataset could actually contradict (Day 2's Hansen `lossyear` layer shows deforestation
is already happening on Negros), so the future map below should be read as "if climate changes but
forest cover doesn't" — not as a full forecast.

In [ ]:
# --- Derive BIO5/BIO14-equivalents from the future climate file (max/min across its 12 monthly values) ---
clim_future = xr.open_dataset(PROCESSED_DIR / 'climate_philippines_2071_2099_ssp460_mean.nc')
bio5_future  = clim_future['air_temperature'].max('time')    # BIO5-equivalent
bio14_future = clim_future['precipitation'].min('time')      # BIO14-equivalent
FUT_LAT, FUT_LON = clim_future['lat'].values, clim_future['lon'].values

# Elevation and forest cover have no "future" -- block-average the same 1 km layers down onto this
# coarser 0.1° grid instead (see the forest-cover caveat above)
elev_coarse  = dem_ds['elevation'].coarsen(lat=12, lon=12, boundary='trim').mean()
treecover_coarse = forest_ds['treecover2000'].coarsen(lat=12, lon=12, boundary='trim').mean()

lsm_0p1 = xr.open_dataset(PROCESSED_DIR / 'land_sea_mask_0p1.nc')
fut_lon_grid, fut_lat_grid = np.meshgrid(FUT_LON, FUT_LAT)
fut_borneo_corner = (fut_lon_grid < 119.0) & (fut_lat_grid < 7.5)
fut_land_mask = (lsm_0p1['land_sea_mask'].values > 0.5) & ~fut_borneo_corner

frows, fcols = np.where(fut_land_mask)
X_future = np.column_stack([
    bio5_future.values[frows, fcols],
    bio14_future.values[frows, fcols],
    elev_coarse.values[frows, fcols],
    treecover_coarse.values[frows, fcols],
])
fvalid = ~np.isnan(X_future).any(axis=1)

future_suitability = np.full(fut_land_mask.shape, np.nan, dtype='float32')
future_suitability[frows[fvalid], fcols[fvalid]] = model.predict_proba(X_future[fvalid])[:, 1]

# --- How much of "present vs future" is real warming vs. just switching data products? ---
clim_present_beck = xr.open_dataset(PROCESSED_DIR / 'climate_philippines_1991_2020_mean.nc')
bio5_present_beck = clim_present_beck['air_temperature'].max('time')

worldclim_mean    = np.nanmean(np.where(land_mask, bio_ds['bio5'].values, np.nan))
beck_present_mean = np.nanmean(np.where(fut_land_mask, bio5_present_beck.values, np.nan))
beck_future_mean  = np.nanmean(np.where(fut_land_mask, bio5_future.values, np.nan))

print('BIO5 (max temp of warmest month), mean over Philippine land:')
print(f'  WorldClim, 1970-2000 (what the model was trained on)      : {worldclim_mean:.1f} degC')
print(f'  Beck ensemble, 1991-2020 ("present", Day 1\'s own baseline): {beck_present_mean:.1f} degC')
print(f'  Beck ensemble, 2071-2099 SSP4-6.0 ("future")               : {beck_future_mean:.1f} degC')
print(f'  -> real warming, same lineage (Beck 1991-2020 -> 2071-2099): {beck_future_mean - beck_present_mean:+.1f} degC')
print(f'  -> apparent "warming" if you naively diff training vs future product: {beck_future_mean - worldclim_mean:+.1f} degC')
print('  These two numbers should NOT be treated as the same thing.')

# --- Extrapolation check: does the future climate fall outside what the model ever saw in training? ---
bio5_train_range  = (sdm_df['bio5'].min(),  sdm_df['bio5'].max())
bio14_train_range = (sdm_df['bio14'].min(), sdm_df['bio14'].max())
frac_out_bio5  = ((X_future[fvalid, 0] < bio5_train_range[0])  | (X_future[fvalid, 0] > bio5_train_range[1])).mean()
frac_out_bio14 = ((X_future[fvalid, 1] < bio14_train_range[0]) | (X_future[fvalid, 1] > bio14_train_range[1])).mean()
print(f'\nBIO5 training range:  {bio5_train_range[0]:.1f}-{bio5_train_range[1]:.1f} degC -- '
      f'{frac_out_bio5:.1%} of future land cells fall outside it')
print(f'BIO14 training range: {bio14_train_range[0]:.1f}-{bio14_train_range[1]:.1f} mm -- '
      f'{frac_out_bio14:.1%} of future land cells fall outside it')

In [ ]:
vmax = max(np.nanmax(suitability), np.nanmax(future_suitability))

fig, axes = plt.subplots(1, 2, figsize=(13, 9))
im0 = axes[0].pcolormesh(PH_LON, PH_LAT, suitability, cmap='viridis', vmin=0, vmax=vmax, shading='auto')
axes[0].set_title('Present (WorldClim 1970–2000, 1 km)')
im1 = axes[1].pcolormesh(FUT_LON, FUT_LAT, future_suitability, cmap='viridis', vmin=0, vmax=vmax, shading='auto')
axes[1].set_title('2071–2099 SSP4-6.0 (Beck ensemble, 0.1° — different source)')

for a in axes:
    a.set_xlim(PH_LON.min(), PH_LON.max())
    a.set_ylim(PH_LAT.min(), PH_LAT.max())
    a.set_aspect('equal')
    a.set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
fig.colorbar(im1, ax=axes, label='Predicted suitability', shrink=0.6)

## *PART IV: How to be a good scientist: tips, tricks, and pitfalls*

**Plan.** Short, discussion-heavy wrap-up (1-2 small demo cells at most) — new pitfalls specific to SDMs,
distinct from Day 2 Part IV's already-covered lessons (season/cloud confounding, single-snapshot vs.
multi-year products). Every bullet below now has a concrete, already-found instance from this dataset
rather than a hypothetical.

- **Presence-only bias, worked example:** the Bohol cluster from Part I. GBIF records cluster near
  tourist/sanctuary sites, not necessarily near the highest-quality habitat — a model trained on biased
  presences learns the bias too. Tarsiers are also famously stress-sensitive to handling, so the very
  popularity that generates GBIF records may itself be a conservation concern worth naming.
- **Your bounding box isn't your country:** the rectangular PH bbox spills into northern Borneo (Sabah) —
  confirmed by the DEM's ~3900 m max elevation matching Mt. Kinabalu. If background/pseudo-absence
  sampling isn't explicitly restricted past the land-sea mask, the model quietly learns from a foreign
  climate. Already handled in Part II, but worth surfacing here as a general lesson: a study-area
  rectangle is a convenience, not a ground truth.
- **Pseudo-absence sensitivity**: briefly show how the fitted suitability map shifts if background points
  are resampled differently (e.g. restricted to a smaller region vs. the whole country) — cheap and
  concrete.
- **Extrapolation / data-lineage risk, two instances:** (1) the classic case — check whether any future
  (2071-2099) grid cells fall outside the range of climate values the model ever saw during training; (2)
  the *new* instance from this dataset — the future map is built from a different climate product and
  resolution (ERA5-derived, 0.1°) than the present-day training data (WorldClim, 1 km, 1970-2000
  baseline), so "present vs. future" isn't purely a climate-change signal, it's partly a data-source
  change. Naming both makes the general lesson ("know what your data actually is") land harder.
- **Correlation vs. causation, again** (echoing Day 1): elevation is a proxy, not a cause — tarsiers don't
  respond to elevation directly, they respond to forest structure and microclimate that happen to
  correlate with it. Climate and terrain correlate with suitability but don't capture roost-site
  specifics, food availability, hunting/tourism pressure, or dispersal limits.
- Close with the same "validate against an independent source" habit taught all workshop: compare the
  model's high-suitability zones against the tarsier's documented range (the Mindanao faunal region —
  Mindanao, Bohol, Leyte, Samar) from the literature.